# 10年定着予測 - 月次系列モデル（GRU/1D-CNN、55_）

## 位置づけ: 単体最強は狙わない。アンサンブルの多様性が目的

これまでの全ノートブック（`18_`〜`53_`）は、月次24ヶ月分のデータを**要約統計に潰してから**
（mean/std/early/late/slope等）personaにマージする方式だった。CatBoostがこの要約統計方式で
LightGBM・XGBoostに明確な差（+0.026〜+0.032）で勝つことは確立済み（[[catboost-beats-other-gbdt]]）。

本ノートブックは、**要約統計を経由せず、24ヶ月の生系列をGRU/1D-CNNに直接食わせる**、
構造的に別物のモデルを作る。データ規模（Train 2761件）を考えると、単体でCatBoostに勝つ見込みは
薄いというのが正直な事前予想。**狙いは「CatBoostとは違う情報・違う誤り方をする、アンサンブルに
値する多様なモデル」を作ること。**

`46_`の教訓に従い、判断はラベルを見ない指標（予測相関・平均絶対差）で先に多様性を確認し、
それがノイズ床（0.02122、[[refit-chaos-noise-floor]]）を超えている場合のみPublicで確認する。
**検証スコアの良し悪しだけでは提出を決めない。**

## 設計

- 入力: 月次9指標 × 24ヶ月の系列（残業時間・有給取得日数・欠勤日数・研修時間・
  上司との面談実施回数・情報共有件数・在宅勤務日数・360度評価5種平均・月例給与）
- 欠損（360度評価は月0-5が構造的欠損、顧客満足度評価・担当プロジェクト数は職種依存で欠損率高い）
  は`-999`で埋め、別チャンネルで欠損マスクを持たせる（CatBoostの`-999`埋めと同じ思想）
- 24ヶ月に満たない社員（早期退職者、Testには存在しない）は末尾を`-999`パディング
- 小さいGRU（hidden=32、1層）+ 最終隠れ状態をpersonaの主要数値特徴量と結合 → 2層MLP → sigmoid
- 学習: 80/20時系列分割・生存者のみ検証（既存プロトコルと同一）、Adamでearly stopping
- ハイパーパラメータ探索はしない（探索コストとモデルの複雑さを釣り合わせない）

> ⚠️ **ローカルMacで先行実行しないこと。**


In [1]:
!pip install -q catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 14.4 MB/s eta 0:00:0000:01:00:01


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（系列モデルもCPUで学習する。データ規模が小さくGPUは不要）")

CPUコア数: 8（系列モデルもCPUで学習する。データ規模が小さくGPUは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import json
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import log_loss
import catboost as cb  # 予測相関の比較用（学習には使わない）

from common.utils.logger import get_logger
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")

SEED = 42
seed_everything(seed=SEED)
torch.manual_seed(SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [5]:
SCRIPT_NAME = "55_sequence_model_gru"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Output Directory: {OUTPUT_DIR}")

[2026-08-15 00:36:50] [INFO] === [55_sequence_model_gru] 実験開始 ===


INFO:55_sequence_model_gru:=== [55_sequence_model_gru] 実験開始 ===


[2026-08-15 00:36:51] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260815


INFO:55_sequence_model_gru:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260815


## 1. データ読み込み・早期退職者の特定（既存ノートブックと同一ロジック）

In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"Train Persona: {train_persona.shape}, Test Persona: {test_persona.shape}")
logger.info(f"定着率: {y_train.mean():.4f}")

EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())
logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / Test: {len(_test_early)}名")
assert len(_test_early) == 0, "Testに早期退職者が存在する。前提が崩れている"

[2026-08-15 00:36:54] [INFO] Train Persona: (2761, 20), Test Persona: (2502, 19)


INFO:55_sequence_model_gru:Train Persona: (2761, 20), Test Persona: (2502, 19)


[2026-08-15 00:36:54] [INFO] 定着率: 0.5647


INFO:55_sequence_model_gru:定着率: 0.5647


[2026-08-15 00:36:54] [INFO] Train 早期退職者: 129名 / Test: 0名


INFO:55_sequence_model_gru:Train 早期退職者: 129名 / Test: 0名


## 2. 月次系列テンソルの構築

24ヶ月に満たない社員は末尾を`-999`パディングし、欠損マスクチャンネルを別に持たせる。

In [7]:
SEQ_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]
SEQ_LEN = 24
MISSING_VALUE = -999.0


def build_sequence_tensor(monthly_df, employee_ids):
    """(N, SEQ_LEN, 2*len(SEQ_METRICS)) のnumpy配列を返す。
    前半チャンネルが実測値(-999埋め)、後半チャンネルが欠損マスク(1=欠損)。"""
    n = len(employee_ids)
    k = len(SEQ_METRICS)
    values = np.full((n, SEQ_LEN, k), MISSING_VALUE, dtype=np.float32)
    mask = np.ones((n, SEQ_LEN, k), dtype=np.float32)

    grouped = {eid: g.sort_values("経過月数") for eid, g in monthly_df.groupby("社員ID")}
    for i, eid in enumerate(employee_ids):
        g = grouped.get(eid)
        if g is None:
            continue
        for j, col in enumerate(SEQ_METRICS):
            col_vals = g[col].values
            valid = ~pd.isna(col_vals)
            idx = np.clip(g["経過月数"].values[valid].astype(int), 0, SEQ_LEN - 1)
            values[i, idx, j] = col_vals[valid].astype(np.float32)
            mask[i, idx, j] = 0.0

    return np.concatenate([values, mask], axis=-1)


logger.info("月次系列テンソルを構築中...")
X_seq_train = build_sequence_tensor(train_monthly, train_ids)
X_seq_test = build_sequence_tensor(test_monthly, test_ids)
logger.info(f"系列テンソル: Train {X_seq_train.shape}, Test {X_seq_test.shape}")
N_CHANNELS = X_seq_train.shape[-1]

[2026-08-15 00:36:54] [INFO] 月次系列テンソルを構築中...


INFO:55_sequence_model_gru:月次系列テンソルを構築中...


[2026-08-15 00:37:00] [INFO] 系列テンソル: Train (2761, 24, 30), Test (2502, 24, 30)


INFO:55_sequence_model_gru:系列テンソル: Train (2761, 24, 30), Test (2502, 24, 30)


## 3. persona側の静的特徴量（数値のみ、シンプルに）

In [8]:
STATIC_NUM_COLS = ["入社時年齢", "前職経験月数", "初任給_円"]


def build_static_features(persona_df):
    X = persona_df[STATIC_NUM_COLS].fillna(MISSING_VALUE).values.astype(np.float32)
    return X


X_static_train = build_static_features(train_persona)
X_static_test = build_static_features(test_persona)

# 系列側と同じスケール感に正規化（学習データの平均・標準偏差でfit）
_static_mean = X_static_train.mean(axis=0)
_static_std = X_static_train.std(axis=0) + 1e-6
X_static_train = (X_static_train - _static_mean) / _static_std
X_static_test = (X_static_test - _static_mean) / _static_std

logger.info(f"静的特徴量: Train {X_static_train.shape}, Test {X_static_test.shape}")

[2026-08-15 00:37:00] [INFO] 静的特徴量: Train (2761, 3), Test (2502, 3)


INFO:55_sequence_model_gru:静的特徴量: Train (2761, 3), Test (2502, 3)


## 4. 時系列split（80/20、検証は生存者のみ。既存プロトコルと同一）

In [9]:
sorted_idx = train_persona.sort_values("入社日").index.values
split_point = int(len(sorted_idx) * 0.8)
train_idx_all = sorted_idx[:split_point]
val_idx_all = sorted_idx[split_point:]

val_ids_all = train_persona.loc[val_idx_all, ID_COL].values
surv_mask = ~pd.Series(val_ids_all).isin(EARLY_LEAVER_IDS).values
val_idx_surv = val_idx_all[surv_mask]

logger.info(f"学習: {len(train_idx_all)}名 / 検証(生存者のみ): {len(val_idx_surv)}名")

y_arr = y_train.values.astype(np.float32)

[2026-08-15 00:37:00] [INFO] 学習: 2208名 / 検証(生存者のみ): 535名


INFO:55_sequence_model_gru:学習: 2208名 / 検証(生存者のみ): 535名


## 5. GRUモデル定義

In [10]:
class SequenceRetentionModel(nn.Module):
    def __init__(self, n_channels, n_static, hidden_size=32, static_hidden=16):
        super().__init__()
        self.gru = nn.GRU(input_size=n_channels, hidden_size=hidden_size,
                           num_layers=1, batch_first=True)
        self.static_fc = nn.Sequential(
            nn.Linear(n_static, static_hidden), nn.ReLU(), nn.Dropout(0.2)
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size + static_hidden, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x_seq, x_static):
        _, h = self.gru(x_seq)
        h = h.squeeze(0)  # (batch, hidden_size)
        s = self.static_fc(x_static)
        z = torch.cat([h, s], dim=-1)
        return self.head(z).squeeze(-1)


class SeqDataset(Dataset):
    def __init__(self, X_seq, X_static, y=None):
        self.X_seq = torch.from_numpy(X_seq)
        self.X_static = torch.from_numpy(X_static)
        self.y = None if y is None else torch.from_numpy(y)

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, i):
        if self.y is None:
            return self.X_seq[i], self.X_static[i]
        return self.X_seq[i], self.X_static[i], self.y[i]


print("✅ モデル定義完了")

✅ モデル定義完了


## 6. 学習関数（early stopping付き）

In [11]:
def train_model(X_seq_tr, X_static_tr, y_tr, X_seq_va, X_static_va, y_va,
                 seed=SEED, max_epochs=60, patience=8, lr=1e-3, batch_size=64):
    """holdout検証つき学習。early stoppingの基準に使ったval_idx_survは、
    Train全件学習では学習データに含まれてしまうため、全件学習では使い回さない
    （37_/40_のfit_full_fixedと同じ思想: holdoutで決めた epoch数を固定値として流用する）。"""
    torch.manual_seed(seed)
    device = torch.device("cpu")

    model = SequenceRetentionModel(N_CHANNELS, X_static_tr.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()

    train_loader = DataLoader(SeqDataset(X_seq_tr, X_static_tr, y_tr),
                               batch_size=batch_size, shuffle=True)
    Xs_va = torch.from_numpy(X_seq_va)
    Xt_va = torch.from_numpy(X_static_va)

    best_val = np.inf
    best_state = None
    best_epoch = 0
    no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        for xb_seq, xb_static, yb in train_loader:
            opt.zero_grad()
            logits = model(xb_seq, xb_static)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xs_va, Xt_va)
            val_proba = torch.sigmoid(val_logits).numpy()
        val_loss = log_loss(y_va, val_proba)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val, best_epoch


def train_model_fixed_epochs(X_seq_tr, X_static_tr, y_tr, n_epochs,
                              seed=SEED, lr=1e-3, batch_size=64):
    """検証セットを使わず、固定エポック数だけ学習する（提出用のTrain全件学習で使う）。"""
    torch.manual_seed(seed)
    model = SequenceRetentionModel(N_CHANNELS, X_static_tr.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()

    train_loader = DataLoader(SeqDataset(X_seq_tr, X_static_tr, y_tr),
                               batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(n_epochs):
        for xb_seq, xb_static, yb in train_loader:
            opt.zero_grad()
            logits = model(xb_seq, xb_static)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

    return model


print("✅ 学習関数定義完了（holdout用 / 全件学習用の2種）")

✅ 学習関数定義完了（holdout用 / 全件学習用の2種）


## 7. holdout検証（複数シード平均、生存者535名で採点）

In [12]:
SEEDS_VAL = [42, 2024, 7, 1234, 99]

val_preds_list = []
best_epochs = []
for seed in SEEDS_VAL:
    model, best_val, best_epoch = train_model(
        X_seq_train[train_idx_all], X_static_train[train_idx_all], y_arr[train_idx_all],
        X_seq_train[val_idx_surv], X_static_train[val_idx_surv], y_arr[val_idx_surv],
        seed=seed,
    )
    model.eval()
    with torch.no_grad():
        proba = torch.sigmoid(model(
            torch.from_numpy(X_seq_train[val_idx_surv]),
            torch.from_numpy(X_static_train[val_idx_surv]),
        )).numpy()
    val_preds_list.append(proba)
    best_epochs.append(best_epoch)
    logger.info(f"  seed={seed}: val_logloss={best_val:.6f}, best_epoch={best_epoch}")

val_preds = np.array(val_preds_list)
y_val_surv = y_arr[val_idx_surv]
val_seedavg_logloss = log_loss(y_val_surv, val_preds.mean(axis=0))
logger.info(f"シード平均 holdout logloss（生存者{len(val_idx_surv)}名）: {val_seedavg_logloss:.6f}")
print(f"GRU系列モデル holdout logloss: {val_seedavg_logloss:.6f}")
print("比較: 49_ R0_memofix(単層CatBoost) holdout logloss = 0.512909")

# Train全件学習で使う固定エポック数（37_/40_のITER_FULLと同じ思想）
FIXED_EPOCHS = max(1, round(np.mean(best_epochs)))
print(f"\nholdoutでの平均最適エポック数: {best_epochs} → 全件学習では{FIXED_EPOCHS}エポックに固定")

[2026-08-15 00:37:18] [INFO]   seed=42: val_logloss=0.649916, best_epoch=29


INFO:55_sequence_model_gru:  seed=42: val_logloss=0.649916, best_epoch=29


[2026-08-15 00:37:32] [INFO]   seed=2024: val_logloss=0.649743, best_epoch=32


INFO:55_sequence_model_gru:  seed=2024: val_logloss=0.649743, best_epoch=32


[2026-08-15 00:37:46] [INFO]   seed=7: val_logloss=0.649545, best_epoch=33


...

[2026-08-15 00:38:26] [INFO]   seed=99: val_logloss=0.650213, best_epoch=58


INFO:55_sequence_model_gru:  seed=99: val_logloss=0.650213, best_epoch=58


[2026-08-15 00:38:26] [INFO] シード平均 holdout logloss（生存者535名）: 0.649248


INFO:55_sequence_model_gru:シード平均 holdout logloss（生存者535名）: 0.649248


GRU系列モデル holdout logloss: 0.649248
比較: 49_ R0_memofix(単層CatBoost) holdout logloss = 0.512909

holdoutでの平均最適エポック数: [29, 32, 33, 58, 58] → 全件学習では42エポックに固定


## 8. 多様性チェック（46_の教訓: ラベルを見ない指標で先に確認）

同じ生存者535名に対する`49_` R0_memofixの検証予測（`_valpreds.npy`）と比較する。

In [13]:
_r0_valpreds_candidates = sorted((PROJECT_ROOT / "data" / "output").glob(
    "*/*_49_memo_parser_fix_R0_memofix_valpreds.npy"))
NOISE_FLOOR_P95 = 0.02122  # 41_実測

if _r0_valpreds_candidates:
    catboost_val_preds = np.load(_r0_valpreds_candidates[-1])  # shape (8シード, N)
    catboost_val_mean = catboost_val_preds.mean(axis=0)
    gru_val_mean = val_preds.mean(axis=0)

    corr = np.corrcoef(catboost_val_mean, gru_val_mean)[0, 1]
    mad = np.abs(catboost_val_mean - gru_val_mean).mean()
    print(f"CatBoost(49_ R0_memofix) vs GRU系列モデル: 相関={corr:.5f} MAD={mad:.5f}（ノイズ床{NOISE_FLOOR_P95}）")
    if mad > NOISE_FLOOR_P95:
        print("✅ ノイズ床を超える多様性あり。ブレンドを試す価値がある")
    else:
        print("⚠️ ノイズ床未満。CatBoostとほぼ同じ予測をしているだけで、多様性としての価値は薄い")

    # 単純平均ブレンドのholdout logloss（重みは学習しない、固定1:1）
    blend = 0.5 * catboost_val_mean + 0.5 * gru_val_mean
    blend_logloss = log_loss(y_val_surv, blend)
    print(f"\n単純平均ブレンド holdout logloss: {blend_logloss:.6f}")
    print(f"CatBoost単体 holdout logloss:     0.512909（49_実測）")
    print(f"GRU単体 holdout logloss:          {val_seedavg_logloss:.6f}")
else:
    print("⚠️ 49_のvalpreds.npyが見つからなかった（多様性チェックをスキップ）")
    catboost_val_mean = None

CatBoost(49_ R0_memofix) vs GRU系列モデル: 相関=0.34584 MAD=0.18912（ノイズ床0.02122）
✅ ノイズ床を超える多様性あり。ブレンドを試す価値がある

単純平均ブレンド holdout logloss: 0.556238
CatBoost単体 holdout logloss:     0.512909（49_実測）
GRU単体 holdout logloss:          0.649248


## 9. Train全件で学習 → Test予測（提出用）

多様性チェックがノイズ床を超えていた場合のみ、この先に進む価値がある。

In [14]:
SEEDS_SUB = [42, 2024, 7, 1234, 99]

test_preds_list = []
for seed in SEEDS_SUB:
    # holdoutで決めたFIXED_EPOCHS分だけ、検証セットを使わずTrain全件で学習する
    # (val_idx_survは全件学習データに含まれるため、早期停止の基準には使えない)
    model = train_model_fixed_epochs(X_seq_train, X_static_train, y_arr, FIXED_EPOCHS, seed=seed)
    model.eval()
    with torch.no_grad():
        proba = torch.sigmoid(model(
            torch.from_numpy(X_seq_test), torch.from_numpy(X_static_test),
        )).numpy()
    test_preds_list.append(proba)
    logger.info(f"  seed={seed}: 全件学習完了（{FIXED_EPOCHS}エポック固定）")

test_preds_gru = np.array(test_preds_list).mean(axis=0)
print(f"GRU系列モデル テスト予測平均: {test_preds_gru.mean():.4f}")

gru_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_gru_single.csv"
pd.DataFrame({ID_COL: test_ids, "pred": test_preds_gru}).to_csv(gru_path, header=False, index=False)
logger.info(f"提出ファイル(GRU単体、参考): {gru_path.name}")

[2026-08-15 00:38:44] [INFO]   seed=42: 全件学習完了（42エポック固定）


INFO:55_sequence_model_gru:  seed=42: 全件学習完了（42エポック固定）


[2026-08-15 00:39:01] [INFO]   seed=2024: 全件学習完了（42エポック固定）


INFO:55_sequence_model_gru:  seed=2024: 全件学習完了（42エポック固定）


[2026-08-15 00:39:18] [INFO]   seed=7: 全件学習完了（42エポック固定）


...

INFO:55_sequence_model_gru:  seed=1234: 全件学習完了（42エポック固定）


[2026-08-15 00:39:52] [INFO]   seed=99: 全件学習完了（42エポック固定）


INFO:55_sequence_model_gru:  seed=99: 全件学習完了（42エポック固定）


GRU系列モデル テスト予測平均: 0.5700
[2026-08-15 00:39:52] [INFO] 提出ファイル(GRU単体、参考): 20260815_55_sequence_model_gru_gru_single.csv


INFO:55_sequence_model_gru:提出ファイル(GRU単体、参考): 20260815_55_sequence_model_gru_gru_single.csv


## 10. ブレンド提出ファイルの作成（多様性がノイズ床を超えていた場合のみ）

現最良（`50_` AG50_full441_weighted, Public 0.513108）とGRUの単純平均（固定1:1、重みは学習しない）。

In [15]:
_best_files = sorted((PROJECT_ROOT / "data" / "output").glob(
    "*/*_50_autogluon_memofix_AG50_full441_weighted.csv"))
assert _best_files, "現最良(50_ AG50_full441_weighted)の提出ファイルが見つからない"
best_test_pred = pd.read_csv(_best_files[-1], header=None, names=[ID_COL, "pred"]).set_index(ID_COL)["pred"]
best_test_pred = best_test_pred.loc[test_ids].values

corr_test = np.corrcoef(best_test_pred, test_preds_gru)[0, 1]
mad_test = np.abs(best_test_pred - test_preds_gru).mean()
print(f"現最良(50_) vs GRU（Test側）: 相関={corr_test:.5f} MAD={mad_test:.5f}（ノイズ床{NOISE_FLOOR_P95}）")

if mad_test > NOISE_FLOOR_P95:
    blend_test = 0.5 * best_test_pred + 0.5 * test_preds_gru
    blend_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_blend_50w_gru.csv"
    pd.DataFrame({ID_COL: test_ids, "pred": blend_test}).to_csv(blend_path, header=False, index=False)
    print(f"✅ ブレンド提出ファイルを作成: {blend_path.name}（予測平均={blend_test.mean():.4f}）")
    print("   このファイルをPublicで確認する価値がある")
else:
    print("⚠️ Test側でもノイズ床未満。ブレンドは作るが、期待値は低いと考えるべき")
    blend_test = 0.5 * best_test_pred + 0.5 * test_preds_gru
    blend_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_blend_50w_gru.csv"
    pd.DataFrame({ID_COL: test_ids, "pred": blend_test}).to_csv(blend_path, header=False, index=False)
    print(f"   （参考用に作成はした: {blend_path.name}）")

現最良(50_) vs GRU（Test側）: 相関=0.30878 MAD=0.22355（ノイズ床0.02122）
✅ ブレンド提出ファイルを作成: 20260815_55_sequence_model_gru_blend_50w_gru.csv（予測平均=0.5781）
   このファイルをPublicで確認する価値がある


## 11. 判定

- **採否はPublicのみ。** holdout loglossの良し悪しでは判断しない（GRU単体はCatBoostに
  勝てない可能性が高いが、それ自体は不採用の理由にならない——目的はアンサンブル価値）
- 第8節・第10節の**予測相関・MADがノイズ床を超えているか**を最優先で確認する。
  超えていなければ、GRU単体もブレンドも提出を見送ってよい
- 超えていれば、ブレンドファイルを1回だけPublicで確認する

> ⚠️ **期待値は控えめに。** データ規模2761件は深層学習の系列モデルには小さく、
> 単体で価値を出すことは最初から狙っていない。

In [16]:
for w_gru in [0.05, 0.10, 0.15, 0.20, 0.30]:
    blend = (1 - w_gru) * catboost_val_mean + w_gru * gru_val_mean
    print(f"w_gru={w_gru}: holdout logloss={log_loss(y_val_surv, blend):.6f}")

w_gru=0.05: holdout logloss=0.514422
w_gru=0.1: holdout logloss=0.516758
w_gru=0.15: holdout logloss=0.519794
w_gru=0.2: holdout logloss=0.523453
w_gru=0.3: holdout logloss=0.532427
